In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("DART_API_KEY")

# DART CORPCODE.xml 다운로드 URL
url = "https://opendart.fss.or.kr/api/corpCode.xml"
params = {"crtfc_key": api_key}

print("CORPCODE.xml 다운로드 중...")
response = requests.get(url, params=params)
print(f"응답 코드: {response.status_code}")
print(f"파일 크기: {len(response.content):,} bytes ({len(response.content) / 1024 / 1024:.2f} MB)")
print(f"Content-Type: {response.headers.get('Content-Type')}")

CORPCODE.xml 다운로드 중...
응답 코드: 200
파일 크기: 3,579,368 bytes (3.41 MB)
Content-Type: application/x-msdownload;charset=UTF-8


In [2]:
import zipfile
import io
import xml.etree.ElementTree as ET

# response.content는 ZIP 바이너리
# 디스크에 저장하지 않고 메모리에서 바로 압축 해제
zf = zipfile.ZipFile(io.BytesIO(response.content))

# ZIP 안에 있는 파일 목록 확인
print("ZIP 내부 파일:", zf.namelist())

# CORPCODE.xml 읽기
xml_data = zf.read("CORPCODE.xml")
print(f"XML 크기: {len(xml_data):,} bytes ({len(xml_data) / 1024 / 1024:.2f} MB)")

# XML 첫 500자 미리보기
print("\n=== XML 미리보기 ===")
print(xml_data[:500].decode("utf-8"))

ZIP 내부 파일: ['CORPCODE.xml']
XML 크기: 29,912,902 bytes (28.53 MB)

=== XML 미리보기 ===
<?xml version="1.0" encoding="UTF-8"?>
<result>
    <list>
        <corp_code>00434003</corp_code>
        <corp_name>다코</corp_name>
        <corp_eng_name>Daco corporation</corp_eng_name>
        <stock_code> </stock_code>
        <modify_date>20170630</modify_date>
    </list>
    <list>
        <corp_code>00430964</corp_code>
        <corp_name>굿앤엘에스</corp_name>
        <corp_eng_name>Good &amp; LS Co.,Ltd.</corp_eng_name>
        <stock_code> </stock_code>
        <modify_date>


In [3]:
import pandas as pd

# XML 파싱
root = ET.fromstring(xml_data)

# 각 <list> 요소에서 필요한 필드 추출
rows = []
for child in root.iter("list"):
    rows.append({
        "corp_code": child.findtext("corp_code"),
        "corp_name": child.findtext("corp_name"),
        "stock_code": child.findtext("stock_code"),
        "modify_date": child.findtext("modify_date"),
    })

# DataFrame으로 변환
df_all = pd.DataFrame(rows)

print(f"전체 기업 수: {len(df_all):,}개")
print(f"\n컬럼: {list(df_all.columns)}")
print(f"\n=== 상위 5개 ===")
print(df_all.head())

전체 기업 수: 118,145개

컬럼: ['corp_code', 'corp_name', 'stock_code', 'modify_date']

=== 상위 5개 ===
  corp_code          corp_name stock_code modify_date
0  00434003                 다코               20170630
1  00430964              굿앤엘에스               20170630
2  00388953  크레디피아제이십오차유동화전문회사               20170630
3  00179984             연방건설산업               20170630
4  00420143     브룩스피알아이오토메이션잉크               20170630


In [4]:
# 종목코드가 공백이 아닌 것만 = 상장사
listed = df_all[df_all["stock_code"].str.strip() != ""].reset_index(drop=True)

print(f"상장사 수: {len(listed):,}개")
print(f"\n=== 상장사 상위 10개 ===")
print(listed.head(10))

# 우리가 잘 아는 삼성전자 확인
samsung = listed[listed["corp_name"] == "삼성전자"]
print(f"\n=== 삼성전자 확인 ===")
print(samsung)

상장사 수: 3,967개

=== 상장사 상위 10개 ===
  corp_code corp_name stock_code modify_date
0  00260985      한빛네트     036720    20170630
1  00264529      엔플렉스     040130    20170630
2  00358545    동서정보기술     055000    20170630
3  00231567     애드모바일     032600    20170630
4  00359614       리더컴     056140    20170630
5  00153551    허메스홀딩스     012400    20170630
6  00344746      유티엑스     045880    20170630
7  00261188     글로포스트     037830    20170630
8  00268020      쏠라엔텍     030390    20170630
9  00269287        보홍     041320    20170630

=== 삼성전자 확인 ===
     corp_code corp_name stock_code modify_date
3374  00126380      삼성전자     005930    20251201


In [ ]:
from pykrx import stock
from datetime import datetime, timedelta

def get_recent_business_day(max_days_back=10):
    """KRX 데이터가 있는 가장 최근 영업일을 찾는다.
    
    오늘부터 거꾸로 최대 10일까지 시도.
    """
    today = datetime.now()
    for days_back in range(max_days_back):
        date_str = (today - timedelta(days=days_back)).strftime("%Y%m%d")
        try:
            # 적은 데이터로 테스트 (KOSPI 한 번만)
            tickers = stock.get_market_ticker_list(date=date_str, market="KOSPI")
            if tickers and len(tickers) > 0:
                return date_str, len(tickers)
        except Exception:
            continue  # 휴장일이면 IndexError 발생 → 다음 날짜로
    
    raise RuntimeError(f"최근 {max_days_back}일 안에 영업일을 찾지 못했습니다")


# 최근 영업일 찾기
print("최근 영업일 탐색 중...")
business_day, kospi_count = get_recent_business_day()
print(f"✅ 사용할 기준일: {business_day} (KOSPI {kospi_count}개 확인)")

# 그 날짜로 양쪽 시장 종목 조회
print("\nKOSPI/KOSDAQ 종목 목록 가져오는 중...")
kospi_codes = set(stock.get_market_ticker_list(date=business_day, market="KOSPI"))
kosdaq_codes = set(stock.get_market_ticker_list(date=business_day, market="KOSDAQ"))

print(f"KOSPI: {len(kospi_codes):,}개")
print(f"KOSDAQ: {len(kosdaq_codes):,}개")

# 시장 구분 함수
def get_market(code):
    if code in kospi_codes:
        return "KOSPI"
    elif code in kosdaq_codes:
        return "KOSDAQ"
    else:
        return "OTHER"  # KONEX, 상폐 임박 등

# 매핑 테이블에 market 컬럼 추가
listed["market"] = listed["stock_code"].apply(get_market)

# 시장별 분포 확인
print(f"\n=== 시장별 분포 ===")
print(listed["market"].value_counts())

ModuleNotFoundError: No module named 'pykrx'

In [ ]:
import sys
print("Python 경로:", sys.executable)

In [ ]:
from pykrx import stock
from datetime import datetime, timedelta
import traceback

# 어떤 에러가 나는지 정확히 보기 위해 try/except를 풀어서 진단
print("=" * 60)
print("진단 시작 — pykrx 직접 호출 테스트")
print("=" * 60)

# 1️⃣ 오늘 날짜로 시도
today = datetime.now().strftime("%Y%m%d")
print(f"\n[1] 오늘({today})로 시도:")
try:
    tickers = stock.get_market_ticker_list(date=today, market="KOSPI")
    print(f"    결과: {len(tickers)}개")
except Exception as e:
    print(f"    에러 타입: {type(e).__name__}")
    print(f"    에러 메시지: {e}")

# 2️⃣ 어제 날짜
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
print(f"\n[2] 어제({yesterday})로 시도:")
try:
    tickers = stock.get_market_ticker_list(date=yesterday, market="KOSPI")
    print(f"    결과: {len(tickers)}개")
except Exception as e:
    print(f"    에러 타입: {type(e).__name__}")
    print(f"    에러 메시지: {e}")

# 3️⃣ 확실한 평일 (지난주 수요일)
past_wed = (datetime.now() - timedelta(days=9)).strftime("%Y%m%d")
print(f"\n[3] 9일 전({past_wed})으로 시도:")
try:
    tickers = stock.get_market_ticker_list(date=past_wed, market="KOSPI")
    print(f"    결과: {len(tickers)}개")
except Exception as e:
    print(f"    에러 타입: {type(e).__name__}")
    print(f"    에러 메시지: {e}")
    traceback.print_exc()

# 4️⃣ pykrx 버전 확인
print(f"\n[4] pykrx 버전:")
import pykrx
print(f"    버전: {pykrx.__version__ if hasattr(pykrx, '__version__') else '알 수 없음'}")

# 5️⃣ 인터넷 연결 확인
print(f"\n[5] KRX 서버 응답 확인:")
import requests
try:
    r = requests.get("https://data.krx.co.kr", timeout=5)
    print(f"    KRX 메인: {r.status_code}")
except Exception as e:
    print(f"    인터넷 연결 실패: {e}")

진단 시작 — pykrx 직접 호출 테스트

[1] 오늘(20260522)로 시도:
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
    결과: 0개

[2] 어제(20260521)로 시도:
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
    결과: 0개

[3] 9일 전(20260513)으로 시도:
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
    결과: 0개

[4] pykrx 버전:
    버전: 1.2.8

[5] KRX 서버 응답 확인:
    KRX 메인: 200
